# Lab 03.1 — HDFS e Spark

**HDFS** (Hadoop Distributed File System) memorizza file molto grandi su più macchine:

* ogni file è diviso in **blocchi** (qui 16 MB, in produzione 128 MB);
* ogni blocco è **replicato** su più **DataNode** (qui 2 copie) → tolleranza ai guasti;
* il **NameNode** conserva solo i metadati: albero delle directory e posizione dei blocchi.

Spark legge HDFS in parallelo: **un blocco → una partizione → un task**, eseguito possibilmente sul nodo che ha il blocco (*data locality*).

Web UI del NameNode: http://localhost:9870 (menu *Utilities → Browse the file system*).

In [ ]:
import os, requests
from pyspark.sql import functions as F
from sessione import crea_sessione

spark = crea_sessione("Lab03-HDFS")
sc = spark.sparkContext
print("Master:", sc.master)
print("Filesystem di default:", sc._jsc.hadoopConfiguration().get("fs.defaultFS"))

## 1. Caricare un file su HDFS tramite WebHDFS (API REST)
La scrittura avviene in due passi: il NameNode risponde con un **redirect** verso il DataNode che riceverà i dati.

In [ ]:
WEBHDFS = "http://namenode:9870/webhdfs/v1"
UTENTE = {"user.name": "jovyan"}

requests.put(f"{WEBHDFS}/dati", params={**UTENTE, "op": "MKDIRS"}).raise_for_status()

locale = "/home/jovyan/datasets/alice_in_wonderland.txt"
r1 = requests.put(f"{WEBHDFS}/dati/alice.txt", params={**UTENTE, "op": "CREATE", "overwrite": "true"},
                  allow_redirects=False)
print("1) NameNode risponde", r1.status_code, "→ redirect a", r1.headers["Location"].split("?")[0])
with open(locale, "rb") as f:
    r2 = requests.put(r1.headers["Location"], data=f)
print("2) DataNode risponde", r2.status_code, "(Created)")

stato = requests.get(f"{WEBHDFS}/dati/alice.txt", params={**UTENTE, "op": "GETFILESTATUS"}).json()["FileStatus"]
print({k: stato[k] for k in ("length", "replication", "blockSize", "owner")})

## 2. La API FileSystem di Hadoop (dalla JVM di Spark)
È la stessa API usata internamente da Spark: funziona con qualunque filesystem compatibile Hadoop (`hdfs://`, `ofs://`, `s3a://`, `file://`).

In [ ]:
jvm = sc._jvm
conf = sc._jsc.hadoopConfiguration()
Path = jvm.org.apache.hadoop.fs.Path
fs = jvm.org.apache.hadoop.fs.FileSystem.get(conf)

def ls(percorso):
    for st in fs.listStatus(Path(percorso)):
        tipo = "d" if st.isDirectory() else "-"
        print(f"{tipo} {st.getReplication()} {st.getLen():>12,d}  {st.getPath().toUri().getPath()}")

ls("/dati")

## 3. Blocchi e repliche
Scriviamo con Spark un file più grande (~ 70 MB) e osserviamo come HDFS lo divide in blocchi da 16 MB replicati su DataNode diversi.

In [ ]:
grande = (spark.range(0, 1_000_000)
          .select(F.col("id"),
                  F.sha2(F.col("id").cast("string"), 256).alias("hash"))
          .coalesce(1))                                  # un solo file, per vedere i blocchi
grande.write.mode("overwrite").option("header", True).csv("/dati/grande")   # path senza schema → HDFS

file_csv = [st.getPath() for st in fs.listStatus(Path("/dati/grande")) if st.getPath().getName().startswith("part-")][0]
st = fs.getFileStatus(file_csv)
print(f"{file_csv.getName()}: {st.getLen() / 1e6:.1f} MB, blocco {st.getBlockSize() // 2**20} MB, replica {st.getReplication()}\n")
for i, b in enumerate(fs.getFileBlockLocations(st, 0, st.getLen())):
    print(f"blocco {i}: offset {b.getOffset() / 1e6:6.1f} MB  lunghezza {b.getLength() / 1e6:5.1f} MB  → DataNode {list(b.getHosts())}")

### Un blocco = una partizione
Leggendo il file con la API RDD, Spark crea una partizione (e quindi un task) per ogni blocco HDFS:
il parallelismo cresce con la dimensione del file.

La API DataFrame raggruppa i blocchi in partizioni da `spark.sql.files.maxPartitionBytes` (default 128 MB).

In [ ]:
righe = sc.textFile("hdfs://namenode:8020/dati/grande")
print("RDD       → partizioni:", righe.getNumPartitions(), "(= blocchi HDFS)")

df = spark.read.option("header", True).csv("/dati/grande")
print("DataFrame → partizioni:", df.rdd.getNumPartitions(), "(maxPartitionBytes = 128 MB)")

spark.conf.set("spark.sql.files.maxPartitionBytes", "16m")
df = spark.read.option("header", True).csv("/dati/grande")
print("DataFrame → partizioni:", df.rdd.getNumPartitions(), "(maxPartitionBytes = 16 MB)")
print("Righe:", df.count())

## 4. Word Count (MapReduce) con input e output su HDFS

In [ ]:
import re
testo = sc.textFile("hdfs://namenode:8020/dati/alice.txt")
conteggi = (testo.flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))
                 .map(lambda w: (w, 1))
                 .reduceByKey(lambda a, b: a + b))
print(conteggi.takeOrdered(10, key=lambda kv: -kv[1]))

fs.delete(Path("/dati/wordcount"), True)
conteggi.map(lambda kv: f"{kv[0]}\t{kv[1]}").saveAsTextFile("/dati/wordcount")
ls("/dati/wordcount")

## 5. Parquet partizionato su HDFS e Spark SQL
Gli stessi concetti del Lab 02 valgono su un filesystem distribuito: cambia solo il percorso.

In [ ]:
vendite = (spark.range(0, 1_000_000)
           .withColumn("negozio", (F.col("id") % 20).cast("int"))
           .withColumn("regione", F.element_at(F.array(*[F.lit(r) for r in ["Nord", "Centro", "Sud", "Isole"]]),
                                               (F.col("id") % 4 + 1).cast("int")))
           .withColumn("importo", F.round(F.rand(7) * 100, 2)))

vendite.write.partitionBy("regione").mode("overwrite").parquet("/dati/vendite")
ls("/dati/vendite")

spark.read.parquet("/dati/vendite").createOrReplaceTempView("vendite")
spark.sql("""
    SELECT regione, COUNT(*) AS scontrini, CAST(SUM(importo) AS DECIMAL(12, 2)) AS totale
    FROM vendite WHERE regione IN ('Sud', 'Isole')
    GROUP BY regione
""").show()

## 6. File locali vs HDFS
Con `fs.defaultFS=hdfs://namenode:8020` un percorso **senza schema** punta a HDFS.
Per i file del container serve lo schema esplicito `file://` (e in un cluster il file deve esistere su **tutti** i nodi).

In [ ]:
print(spark.read.text("file:///home/jovyan/datasets/alice_in_wonderland.txt").count(), "righe dal filesystem locale")
print(spark.read.text("/dati/alice.txt").count(), "righe da HDFS")

In [ ]:
spark.stop()